# Tramos

Cada fila de `data/processed/tramos.csv` es un tramo de calle del microcentro. Acá se miran las columnas, sin entrenar ningún modelo.

`tipo_zona` sale de tags de OpenStreetMap. En este recorte hay residencial y comercial. Industrial no aparece. `prioridad_recoleccion` es sintética.

In [1]:
from pathlib import Path

import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data/processed/tramos.csv").is_file():
            return candidate
    raise FileNotFoundError("no está data/processed/tramos.csv; corré el notebook desde el repo")

ROOT = repo_root()
ZONA = "tipo_zona"
PRIORIDAD = "prioridad_recoleccion"
NUMERICAS = [
    "cantidad_contenedores",
    "dias_desde_ultima_recoleccion",
    "historico_quejas_vecinales",
]
ZONAS = ["residencial", "comercial", "industrial"]
CLASES = ["alta", "media", "baja"]

tramos = pd.read_csv(ROOT / "data/processed/tramos.csv")
presentes = [zona for zona in ZONAS if (tramos[ZONA] == zona).any()]
tramos.shape

(1257, 11)

## Una fila

Las columnas del dataset mínimo, más la calle. `tipo_zona` es una categoría del tramo, no un número.

In [2]:
tramos[["tramo_id", "nombre_calle", ZONA, *NUMERICAS, PRIORIDAD]].head(8)

,tramo_id,nombre_calle,tipo_zona,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales,prioridad_recoleccion
0,1,Teniente General Juan Domingo Perón,residencial,0,4,2,media
1,2,Avenida 9 de Julio,residencial,0,4,1,media
2,3,Avenida 9 de Julio,residencial,0,5,1,media
3,4,Teniente General Juan Domingo Perón,residencial,0,1,0,baja
4,5,Viamonte,residencial,0,7,0,alta
5,6,Avenida 9 de Julio,residencial,0,1,1,baja
6,7,Avenida 9 de Julio,residencial,0,4,1,media
7,8,Viamonte,residencial,0,2,1,baja


## Tipo de zona

Cuántos tramos hay de cada tipo, y cómo se reparte la prioridad adentro de cada uno. Industrial figura en el conteo aunque no haya ninguno. Las tablas de más abajo usan solo las zonas que aparecen.

In [3]:
zonas = tramos[ZONA].value_counts().reindex(ZONAS, fill_value=0)
pd.DataFrame({"tramos": zonas, "proporcion": (zonas / len(tramos)).round(3)})

,tramos,proporcion
tipo_zona,,
residencial,859,0.683
comercial,398,0.317
industrial,0,0.000


In [4]:
por_zona = pd.crosstab(tramos[ZONA], tramos[PRIORIDAD]).reindex(index=ZONAS, columns=CLASES, fill_value=0)
por_zona.loc["total"] = por_zona.sum()
por_zona

prioridad_recoleccion,alta,media,baja
tipo_zona,,,
residencial,150,294,415
comercial,101,146,151
industrial,0,0,0
total,251,440,566


In [5]:
pd.crosstab(tramos[ZONA], tramos[PRIORIDAD], normalize="index").reindex(
    index=presentes, columns=CLASES
).round(3)

prioridad_recoleccion,alta,media,baja
tipo_zona,,,
residencial,0.175,0.342,0.483
comercial,0.254,0.367,0.379


Contenedores, días y quejas según el tipo de zona.

In [6]:
tramos.groupby(ZONA)[NUMERICAS].mean().reindex(presentes).round(2)

,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales
tipo_zona,,,
residencial,0.43,3.20,1.09
comercial,1.31,2.18,1.88


In [7]:
tramos.groupby(ZONA)[NUMERICAS].agg(["min", "median", "max"]).reindex(presentes)

cantidad_contenedores            dias_desde_ultima_recoleccion            historico_quejas_vecinales           
                              min median max                           min median max                        min median max
tipo_zona                                                                                                                  
residencial                     0    0.0   6                             0    3.0  10                          0    1.0   6
comercial                       0    1.0   9                             0    2.0   7                          0    2.0   9

Dos tramos de cada zona que sí aparece, para ver la categoría al lado de los números.

In [8]:
ejemplos = pd.concat(
    [tramos[tramos[ZONA] == zona].head(2) for zona in presentes],
    ignore_index=True,
)
ejemplos[["tramo_id", "nombre_calle", ZONA, *NUMERICAS, PRIORIDAD]]

,tramo_id,nombre_calle,tipo_zona,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales,prioridad_recoleccion
0,1,Teniente General Juan Domingo Perón,residencial,0,4,2,media
1,2,Avenida 9 de Julio,residencial,0,4,1,media
2,17,Avenida Santa Fe,comercial,2,1,2,media
3,18,Suipacha,comercial,0,2,2,media


## Prioridad

La etiqueta queda cerca de alta 20 %, media 35 % y baja 45 %. La media de las otras columnas sube de baja a alta, y los rangos se solapan. La última tabla parte esa media por tipo de zona.

In [9]:
prioridad = tramos[PRIORIDAD].value_counts().reindex(CLASES)
pd.DataFrame({"tramos": prioridad, "proporcion": (prioridad / len(tramos)).round(3)})

,tramos,proporcion
prioridad_recoleccion,,
alta,251,0.20
media,440,0.35
baja,566,0.45


In [10]:
tramos.groupby(PRIORIDAD)[NUMERICAS].mean().reindex(CLASES).round(2)

,cantidad_contenedores,dias_desde_ultima_recoleccion,historico_quejas_vecinales
prioridad_recoleccion,,,
alta,1.84,4.87,2.17
media,0.72,3.25,1.53
baja,0.19,1.70,0.83


In [11]:
tramos.groupby([ZONA, PRIORIDAD])[NUMERICAS].mean().reindex(
    pd.MultiIndex.from_product([presentes, CLASES], names=[ZONA, PRIORIDAD])
).round(2)

cantidad_contenedores  dias_desde_ultima_recoleccion  historico_quejas_vecinales
tipo_zona   prioridad_recoleccion                                                                                  
residencial alta                                    1.25                           5.67                        1.81
            media                                   0.43                           3.76                        1.25
            baja                                    0.13                           1.91                        0.72
comercial   alta                                    2.71                           3.67                        2.69
            media                                   1.31                           2.22                        2.09
            baja                                    0.36                           1.13                        1.13